In [8]:
import pandas as pd

# -----------------------------
# 1. Load the data for individual years
# -----------------------------

# 2024
crash_2024 = pd.read_csv("Data\\Crash_2024.csv", dtype={"CRN": str})
vehicle_2024 = pd.read_csv("Data\\Vehicle_2024.csv", dtype={"CRN": str})
flags_2024 = pd.read_csv("Data\\Flags_2024.csv", dtype={"CRN": str})

# 2023
crash_2023 = pd.read_csv("Data\\Crash_2023.csv", dtype={"CRN": str})
vehicle_2023 = pd.read_csv("Data\\Vehicle_2023.csv", dtype={"CRN": str})
flags_2023 = pd.read_csv("Data\\Flags_2023.csv", dtype={"CRN": str})

# 2022
crash_2022 = pd.read_csv("Data\\Crash_2022.csv", dtype={"CRN": str})
vehicle_2022 = pd.read_csv("Data\\Vehicle_2022.csv", dtype={"CRN": str})
flags_2022 = pd.read_csv("Data\\Flags_2022.csv", dtype={"CRN": str})

# 2021
crash_2021 = pd.read_csv("Data\\Crash_2021.csv", dtype={"CRN": str})
vehicle_2021 = pd.read_csv("Data\\Vehicle_2021.csv", dtype={"CRN": str})
flags_2021 = pd.read_csv("Data\\Flags_2021.csv", dtype={"CRN": str})

# 2020
crash_2020 = pd.read_csv("Data\\Crash_2020.csv", dtype={"CRN": str})
vehicle_2020 = pd.read_csv("Data\\Vehicle_2020.csv", dtype={"CRN": str})
flags_2020 = pd.read_csv("Data\\Flags_2020.csv", dtype={"CRN": str})

In [9]:
# Union the years 2020-2024 using pd.concat
crash = pd.concat([crash_2020, crash_2021, crash_2022, crash_2023, crash_2024], ignore_index=True).drop_duplicates()
vehicle = pd.concat([vehicle_2020, vehicle_2021, vehicle_2022, vehicle_2023, vehicle_2024], ignore_index=True).drop_duplicates()
flags = pd.concat([flags_2020, flags_2021, flags_2022, flags_2023, flags_2024], ignore_index=True).drop_duplicates()

In [10]:
# -----------------------------
# 2. Standardize column names
# -----------------------------
crash.columns = crash.columns.str.upper().str.strip()
vehicle.columns = vehicle.columns.str.upper().str.strip()
flags.columns = flags.columns.str.upper().str.strip()


In [11]:
# -----------------------------
# 3. Merge Crash + Flags (1:1)
# -----------------------------
crash_flags = crash.merge(flags, on="CRN", how="left")

In [12]:
# -----------------------------
# 4. Build dynamic aggregation dictionary
# -----------------------------
agg_dict = {
    "UNIT_NUM": "count",                 # number of vehicles
    "VEH_TYPE": lambda x: x.nunique()    # number of unique vehicle types
}

# Optional fields — only add if they exist
optional_fields = {
    "COMMERCIAL_VEHICLE": "sum",
    "HEAVY_TRUCK": "sum",
    "BUS": "sum",
    "MOTORCYCLE": "sum",
    "BICYCLE": "sum"
}

for col, func in optional_fields.items():
    if col in vehicle.columns:
        agg_dict[col] = func

In [13]:
# -----------------------------
# 5. Aggregate Vehicle table
# -----------------------------
vehicle_agg = (
    vehicle.groupby("CRN")
    .agg(agg_dict)
    .rename(columns={
        "UNIT_NUM": "VEHICLE_COUNT",
        "VEH_TYPE": "UNIQUE_VEHICLE_TYPES"
    })
    .reset_index()
)

In [14]:
# -----------------------------
# 6. Merge Crash+Flags with Vehicle aggregates
# -----------------------------
df_merged = crash_flags.merge(vehicle_agg, on="CRN", how="left")

In [15]:
# -----------------------------
# 7. Final checks
# -----------------------------
print(df_merged.shape)
print(df_merged.head())

(560396, 230)
          CRN  ARRIVAL_TM  AUTOMOBILE_COUNT  BELTED_DEATH_COUNT  \
0  2019133973         NaN                 2                   0   
1  2020006647      1511.0                 1                   0   
2  2020024016       638.0                 1                   0   
3  2020029257      1957.0                 1                   0   
4  2020008250      1950.0                 1                   0   

   BELTED_SUSP_SERIOUS_INJ_COUNT  BICYCLE_COUNT  BICYCLE_DEATH_COUNT  \
0                              0              0                    0   
1                              0              0                    0   
2                              0              0                    0   
3                              0              0                    0   
4                              0              0                    0   

   BICYCLE_SUSP_SERIOUS_INJ_COUNT  BUS_COUNT  CHLDPAS_DEATH_COUNT  ...  URBAN  \
0                               0          0                    0  ..

In [16]:
missing = (
    df_merged.isna()
    .mean()
    .sort_values(ascending=False)
    .to_frame("missing_pct")
)
missing.head(20)


,missing_pct
ILLUMINATION,1.000000
LANE_CLOSED,1.000000
EST_HRS_CLOSED,0.999656
WZ_CLOSE_DETOUR,0.994875
WZ_FLAGGER,0.994727
WZ_MOVING,0.994620
WZ_OTHER,0.994101
WZ_SHLDER_MDN,0.992641
WZ_LN_CLOSURE,0.992598
RDWY_SURF_TYPE_CD,0.990196


In [19]:
df_merged.size

128891080